# Crivo: retenção e hipóteses

Este caderno carrega os pesos próprios já treinados. Não exige repetir treino, GPU ou serviço de modelo externo. Os treinos e os limites estão registrados abaixo; o candidato é experimental e não altera automaticamente o Crivo do chat.

O objetivo desta rodada é interpretar correções e hipóteses, conservar fontes e voltar aos fatos. Isso usa interpretação neural, cópia limitada e cálculo fechado. Não é geração de conversa livre.


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/CRIVO-retencao')
BRANCH='codex/correcoes-hipoteses-20261008'
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/HROSONE/CRIVO.git',str(REPO)])
else:
    b=subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
    if b!=BRANCH:raise RuntimeError('Pasta de outra branch; escolha uma pasta nova.')
print('Revisão:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
LAB=REPO/'experimentos/retencao_contrastes_20261008'
EVENTOS=REPO/'experimentos/escopo_eventos_20261008'


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.2,<3','numpy>=1.24,<3','tokenizers>=0.20,<1'])


## O que os testes mediram

O novo teste foi preparado antes do treino e não escolhe os pesos. Os painéis antigos servem para medir perdas de capacidades já treinadas. A tentativa mistura dados novos e replay dos dois treinos anteriores; preserva as previsões do próprio modelo anterior, sem modelo externo.

As duas tentativas primárias e a composição com escopo antigo também ficam publicadas, com suas falhas. A composição chegou a melhorar um piloto, mas perdeu desempenho em regressões e foi rejeitada. Uma única semente e gramática autoral compartilhada limitam todas essas conclusões.


In [ ]:
r=json.loads((LAB/'avaliacao/resumo.json').read_text())
for nome,paineis in r['resultados'].items():
    print(nome)
    for painel in ['novo','associacao_conhecida','contextual_conhecido']:
        m=paineis[painel+'_limitado']
        print(painel,m['contratos_corretos'],'/',m['n'],'sessões completas:',m['sessoes_completas'],'/',m['sessoes'])
    print('Eventos novos:',{k:(x['contratos_corretos'],x['n']) for k,x in paineis['novo_limitado']['por_evento'].items()})
print('Critérios:',r['criterios'])
print('Passou critério sintético:',r['passou_criterio_sintetico'],'; aprovado para chat:',r['aprovado_para_chat'])
print('Limites:',r['limites'])


## Inspecione falas próprias

Edite as falas abaixo. `MODELO='retencao'` carrega o novo candidato; `MODELO='anterior'` permite comparar. A proposta mostra fontes e se a situação foi interpretada como hipótese. Uma conta executável pode usar uma fonte errada; veja também o histórico e os resultados dos testes.

A gramática cobre preços e requisitos de forma limitada, nomes de uma palavra com inicial maiúscula e até oito nomes candidatos. O histórico deve caber em 256 tokens, sem truncamento. Essas entradas ficam em um arquivo temporário da sua sessão; não são enviadas automaticamente para treino ou Drive. Não há redação livre nesta célula.

Na inspeção local deste exemplo, o quinto turno voltou ao preço 41 depois de a hipótese de 19 ter sido confirmada como fato. Esse é um erro preservado: a resposta correta nesse turno mantém 19. O caderno executou normalmente; o candidato ainda falha semanticamente nesse retorno.


In [ ]:
MODELO='retencao'
FALAS=[
    'O plano Azul custa 23 reais. O plano Verde custa 35 reais. Compare Azul e Verde.',
    'Corrigindo Azul: o custo real é 41 reais. Compare Azul e Verde.',
    'Imagine Azul por 19 reais, sem alterar os fatos. Compare Azul e Verde.',
    'A última hipótese aconteceu de verdade. Agora ela é fato. Compare Azul e Verde.',
    'Esqueça a simulação e use os preços reais corrigidos. Compare Azul e Verde.',
]
p=Path('/content/falas-retencao.json');p.write_text(json.dumps(FALAS,ensure_ascii=False))
RUNNER="""
import json,sys,torch
from pathlib import Path
sys.path.insert(0,sys.argv[1])
from base import ROOT,INICIAL
from rede_eventos import carregar
from normalizacao import codificar
from modelo import lote,executar
from decodificar import proposta_limitada
from tokenizers import Tokenizer
torch.set_num_threads(1)
cp=Path(sys.argv[2]) if sys.argv[4]=='retencao' else INICIAL
m,_=carregar(cp);m.eval()
tok=Tokenizer.from_file(str(ROOT/'artefatos/linguagem_profunda/tokenizer.json'));tok.encode_special_tokens=True
falas=json.loads(Path(sys.argv[3]).read_text())
for i in range(1,len(falas)+1):
    e={'turnos':falas[:i],'argumentos':[None]*3,'referente':None}
    it=codificar(tok,e);x,l=lote([it],tok.token_to_id('<pad>'))
    with torch.no_grad():ls=m(x,l)
    pr=proposta_limitada(it,ls,0)
    print(json.dumps({'turno':i,'proposta':pr,'execucao':executar(pr)},ensure_ascii=False))
"""
subprocess.check_call([sys.executable,'-c',RUNNER,str(EVENTOS),str(LAB/'candidato/pesos.pt'),str(p),MODELO])


## Treino já executado

O professor congelado de retenção foi uma cópia dos pesos anteriores do próprio Crivo. Ele só é necessário durante o treino; a inspeção acima carrega uma rede de aproximadamente 2,95 milhões de parâmetros. Repetir este corpus não equivale a ganhar conversa livre. Scripts e dados permitem reprodução em uma pasta nova, mas este caderno não inicia treino. Foi validado em CPU local; não executado numa sessão real do Colab.
